In [1]:
import torch
import time
import pandas as pd
import torch.nn.utils.prune as prune

from torchvision.models import mobilenet_v2

In [2]:
model = mobilenet_v2(weights="DEFAULT")
model.eval()

print("Model loaded.")

Model loaded.


In [3]:
for module in model.modules():
    if isinstance(module, torch.nn.Conv2d):
        prune.ln_structured(
            module,
            name="weight",
            amount=0.2,
            n=2,
            dim=0
        )

print("20% structured pruning applied.")

20% structured pruning applied.


In [4]:
for module in model.modules():
    if isinstance(module, torch.nn.Conv2d):
        try:
            prune.remove(module, "weight")
        except:
            pass

print("Pruning made permanent.")

Pruning made permanent.


In [5]:
def benchmark(model, device, input_tensor, runs=100):

    model = model.to(device)
    input_tensor = input_tensor.to(device)

    # Warm-up runs
    for _ in range(10):
        with torch.no_grad():
            _ = model(input_tensor)

    # GPU synchronization
    if device == "cuda":
        torch.cuda.synchronize()

    start = time.time()

    for _ in range(runs):
        with torch.no_grad():
            _ = model(input_tensor)

    if device == "cuda":
        torch.cuda.synchronize()

    end = time.time()

    avg_latency = (end - start) / runs

    return avg_latency

In [6]:
input_tensor = torch.randn(1, 3, 224, 224)

In [7]:
cpu_latency = benchmark(model, "cpu", input_tensor)

print(f"CPU Average Latency: {cpu_latency:.6f} seconds")

CPU Average Latency: 0.066158 seconds


In [8]:
gpu_latency = benchmark(model, "cuda", input_tensor)

print(f"GPU Average Latency: {gpu_latency:.6f} seconds")

GPU Average Latency: 0.021111 seconds


In [9]:
results = pd.DataFrame({
    "Device": ["CPU", "GPU"],
    "Latency_Seconds": [cpu_latency, gpu_latency]
})

results.to_csv("pruning_20_results.csv", index=False)

print(results)

  Device  Latency_Seconds
0    CPU         0.066158
1    GPU         0.021111


In [10]:
import os
import torch

torch.save(model.state_dict(), "pruned_model.pth")

size_mb = os.path.getsize("pruned_model.pth") / (1024 * 1024)

print(f"Pruned model size: {size_mb:.2f} MB")

Pruned model size: 13.60 MB


In [11]:
import os
import torch
from torchvision.models import mobilenet_v2

baseline_model = mobilenet_v2(weights="DEFAULT")

torch.save(baseline_model.state_dict(), "baseline_model.pth")

size_mb = os.path.getsize("baseline_model.pth") / (1024 * 1024)

print(f"Baseline model size: {size_mb:.2f} MB")

Baseline model size: 13.60 MB
